# HT/NALM vs HT/HB — B target comparison

Both cell systems share the same **healthy T donor**; the **B target** differs:

* **HT/NALM** = `NALM-6 + healthy T`     — B-ALL cell-line target
* **HT/HB**   = `healthy B + healthy T`  — primary healthy B cells

Because the T compartment is held fixed, the analyses below ask:

1. **T-cell side (CD8):** does the healthy T cell respond differently to the NALM-6 cell line vs primary healthy B cells?
2. **B-cell side:** how does primary healthy B surface biology compare to the NALM-6 model under the same healthy T donor?

Sample availability:

| Time × Condition | HT/NALM | HT/HB |
| --- | --- | --- |
| 6h Mock          | S005 | S001 |
| 6h Blinatumomab  | S006 | S002 |
| 48h Mock         | S007 | S003 |
| 48h Blinatumomab | S008 | S004 |

Both systems have all four time × condition groups, so cross-system comparisons are run at **6h** (consistent with the other paired-system notebooks); 48h subsets are also available where useful.

In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

from nalm_utils import *

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

# Two systems being compared (both use healthy T donor)
SYS_HT_NALM = 'NALM-6 + healthy T'       # HT/NALM — NALM-6 cell-line target
SYS_HT_HB   = 'healthy B + healthy T'    # HT/HB   — primary healthy B

In [ ]:
# [1 · Data loading]
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Sample availability across the two systems being compared
mask_sys = adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB])
print(f'Total cells in HT/NALM + HT/HB: {mask_sys.sum()}')
pd.crosstab(
    index=[adata.obs[mask_sys]['cell_system'], adata.obs[mask_sys]['sample']],
    columns=[adata.obs[mask_sys]['time'], adata.obs[mask_sys]['condition']],
)

In [ ]:
# [1b · UMAP exploration]
mask_explore = (
    (adata.obs['cell_type_annot'].isin(['CD4', 'CD8', 'B'])) &
    (adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB]))
)
adata_sub = adata[mask_explore].copy()

print(f'Cells in HT/NALM + HT/HB (CD4/CD8/B): {adata_sub.n_obs}')
pd.crosstab(index=adata_sub.obs['sample'],
            columns=[adata_sub.obs['cell_system'], adata_sub.obs['cell_type_annot']])

sc.pl.umap(adata_sub,
  color=['CD3e', 'CD19', 'CD8', 'cell_system', 'cell_type_annot', 'condition'],
  layer='arcsinh', frameon=False)

## Cross-condition comparison — CD8 across time × condition

In [ ]:
# [2 · 4-way DA panel: CD8 — HT/NALM vs HT/HB across time × condition]
# B cell markers are dropped to keep the focus on T-cell biology
# (potential B-cell contamination from either NALM-6 or healthy B).
def _load_b_panel():
    for name in ('b_cell_markers', 'or'):
        try:
            return load_marker_panel(name)
        except KeyError:
            continue
    raise KeyError('No B cell marker panel found in marker_panels.json')

B_CELL_MARKERS_SET = set(_load_b_panel())

mask_nm_cd8 = (
    (adata.obs['cell_system'] == SYS_HT_NALM) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_nm_cd8 = adata[mask_nm_cd8].copy()
adata_nm_cd8.obs['time_cond'] = (
    adata_nm_cd8.obs['time'].astype(str) + ' ' + adata_nm_cd8.obs['condition'].astype(str)
)
adata_nm_cd8 = adata_nm_cd8[:, ~adata_nm_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

mask_hb_cd8 = (
    (adata.obs['cell_system'] == SYS_HT_HB) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_hb_cd8 = adata[mask_hb_cd8].copy()
adata_hb_cd8.obs['time_cond'] = (
    adata_hb_cd8.obs['time'].astype(str) + ' ' + adata_hb_cd8.obs['condition'].astype(str)
)
adata_hb_cd8 = adata_hb_cd8[:, ~adata_hb_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

print(f'CD8 HT/NALM: {adata_nm_cd8.n_obs}')
print(adata_nm_cd8.obs['time_cond'].value_counts().to_string())
print(f'\nCD8 HT/HB: {adata_hb_cd8.n_obs}')
print(adata_hb_cd8.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_nm_cd8, 'cd8_t_cell_markers',
    group_key='time_cond',
    adata_compare=adata_hb_cd8,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

In [ ]:
adata.var_names.values

## LFC scatter — Blinatumomab vs Mock at 6h

Both systems have all four time × condition groups; the LFC (Blina/Mock) comparison is shown at **6h** (parallel to the PT/PB vs PT/NALM and HT/NALM vs PT/NALM notebooks).

In [ ]:
# [7 · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), CD8, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD8',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',  # higher LFC in HT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in HT/HB   (below y=x)
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Mock vs Blinatumomab — abundance + spatial, per system

In [ ]:
# [8 · CD8 Blina vs Mock — abundance + spatial, per system, 6h]
# 12c-style 2x2 panel (abundance up/down, spatial coloc up/down) per system.
for sys_label, sys_val in [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='CD8',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## MA plots — LFC vs mean abundance, CD8, 6h

For each feature (CD8 protein abundance markers and spatial colocalization
pairs) we plot the **mean signal** (x, averaged across both conditions in the
system) against the **Blina − Mock LFC** (y). Points are coloured red
(FDR < 0.05, up in Blina), blue (FDR < 0.05, up in Mock) or grey (n.s.); top
features by |LFC| among the significant ones are labelled.

This standardises the abundance × spatial bar plots above by abundance level —
a large LFC at very low mean signal is less convincing than the same LFC at
moderate mean signal, and looking at the trend reveals whether differential
signal is concentrated at low / mid / high abundance.

In [ ]:
# [8b · MA plots — Blina vs Mock LFC vs mean abundance, per system, CD8, 6h]
ma_results_cd8 = plot_ma_blina_vs_mock(
    adata,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='CD8',
)

## ΔLFC vs mean abundance — HT/NALM vs HT/HB, CD8, 6h

*LFC of LFC.* For each CD8 feature we compute the Blina − Mock LFC in each
system separately and then plot:

* **x**: mean signal across the two systems combined (Blina ∪ Mock pooled across
  HT/NALM and HT/HB), as the abundance baseline to standardise against.
* **y**: ΔLFC = LFC(HT/NALM) − LFC(HT/HB) — how much *more* the CD8 response
  in the NALM-6 system exceeds the response with healthy primary B targets.

Color scheme matches the [7 · LFC scatter] above: purple = ↑ in HT/NALM,
green = ↑ in HT/HB. Significance (FDR < 0.05 in either / both sides) is
encoded via marker size and opacity. Because the same healthy T donor is used
in both systems, ΔLFC isolates the contribution of the B target (NALM-6 line
vs primary healthy B) on T-cell behaviour.

In [ ]:
# [8c · ΔLFC (LFC of LFC) vs mean signal — CD8, 6h, NALM vs HB]
delta_ab_cd8, delta_sp_cd8 = plot_delta_lfc_ma(
    ma_results_cd8,
    label_a='HT/NALM',
    label_b='HT/HB',
    cell_type='CD8',
    time_val='6h',
)

## Spatial subsets for selected-marker comparisons

In [ ]:
# [9 · Spatial subsets — CD8 cells, per condition / system]
def _sp_subset(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building spatial subsets (CD8 only):')
sp_6h_mock_nm    = _sp_subset(adata, '6h',  'Mock',         SYS_HT_NALM)
sp_6h_mock_hb    = _sp_subset(adata, '6h',  'Mock',         SYS_HT_HB)
sp_6h_blina_nm   = _sp_subset(adata, '6h',  'Blinatumomab', SYS_HT_NALM)
sp_6h_blina_hb   = _sp_subset(adata, '6h',  'Blinatumomab', SYS_HT_HB)
sp_48h_mock_nm   = _sp_subset(adata, '48h', 'Mock',         SYS_HT_NALM)
sp_48h_mock_hb   = _sp_subset(adata, '48h', 'Mock',         SYS_HT_HB)
sp_48h_blina_nm  = _sp_subset(adata, '48h', 'Blinatumomab', SYS_HT_NALM)
sp_48h_blina_hb  = _sp_subset(adata, '48h', 'Blinatumomab', SYS_HT_HB)

all_sp_cols = sp_6h_mock_nm.columns

## CD8 immune synapse — HT/NALM vs HT/HB at 6h Blinatumomab

In [ ]:
# [12 · CD8 immune synapse: heatmaps + networks at 6h Blina]
SYNAPSE_CATEGORIES = {
    'cSMAC (signaling core)': (['CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137',
                                'CD226', 'TIGIT', 'CD279', 'VISTA'],       '#e41a1c'),
    'pSMAC (adhesion ring)':  (['CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48',
                                'CD352', 'CD53'],                           '#4daf4a'),
    'Exclusion zone':         (['CD45', 'CD43', 'CD44'],                    '#377eb8'),
}

mat_syn_nm, mat_syn_hb, mat_syn_diff = plot_synapse_suite(
    sp_a=sp_6h_blina_nm, sp_b=sp_6h_blina_hb,
    categories=SYNAPSE_CATEGORIES,
    label_a='HT/NALM 6h Blina', label_b='HT/HB 6h Blina',
    diff_label='Diff (HT/NALM − HT/HB)',
    suite_name='Immune synapse',
    highlight_node='CD3e',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)

# B cells — HT/NALM vs HT/HB

The B compartment differs across the two systems: **NALM-6 cell line** (HT/NALM)
vs **primary healthy B cells** (HT/HB), under the shared healthy T donor.
Differences here reflect the gap between the NALM-6 model and primary healthy
B-cell biology rather than a donor effect.

## Cross-condition comparison — B cells across time × condition

In [ ]:
# [B-2 · 4-way DA panel: B cells — HT/NALM vs HT/HB across time × condition]
mask_nm_b = (
    (adata.obs['cell_system'] == SYS_HT_NALM) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_nm_b = adata[mask_nm_b].copy()
adata_nm_b.obs['time_cond'] = (
    adata_nm_b.obs['time'].astype(str) + ' ' + adata_nm_b.obs['condition'].astype(str)
)

mask_hb_b = (
    (adata.obs['cell_system'] == SYS_HT_HB) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_hb_b = adata[mask_hb_b].copy()
adata_hb_b.obs['time_cond'] = (
    adata_hb_b.obs['time'].astype(str) + ' ' + adata_hb_b.obs['condition'].astype(str)
)

print(f'B HT/NALM: {adata_nm_b.n_obs}')
print(adata_nm_b.obs['time_cond'].value_counts().to_string())
print(f'\nB HT/HB: {adata_hb_b.n_obs}')
print(adata_hb_b.obs['time_cond'].value_counts().to_string())

# Use the B-cell panel (currently keyed as 'or' in marker_panels.json).
plot_marker_panel_violins(
    adata_nm_b, 'or',
    group_key='time_cond',
    adata_compare=adata_hb_b,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

# Inhibitory co-markers panel: CD162, CD66b, CD274, CD273.
plot_marker_panel_violins(
    adata_nm_b, 'inhibitory_co_markers',
    group_key='time_cond',
    adata_compare=adata_hb_b,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

## LFC scatter — Blinatumomab vs Mock at 6h, B cells

Both systems have all four time × condition groups; the LFC (Blina/Mock) comparison is shown at **6h** (parallel to the other paired-system notebooks).

In [ ]:
# [B-7 · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), B cells, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='B',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',  # higher LFC in HT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in HT/HB   (below y=x)
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Mock vs Blinatumomab — abundance + spatial, per system, B cells

In [ ]:
# [B-8 · B cell Blina vs Mock — abundance + spatial, per system, 6h]
# 12c-style 2x2 panel (abundance up/down, spatial coloc up/down) per system.
for sys_label, sys_val in [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='B',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## MA plots — LFC vs mean abundance, B cells, 6h

For each feature (protein abundance markers and spatial colocalization pairs)
we plot the **mean signal** (x, averaged across both conditions in the system)
against the **Blina − Mock LFC** (y). Points are coloured red (FDR < 0.05, up
in Blina), blue (FDR < 0.05, up in Mock) or grey (n.s.); top features by |LFC|
among the significant ones are labelled.

This standardises the abundance × spatial bar plots above by abundance level —
a large LFC at very low mean signal is less convincing than the same LFC at
moderate mean signal, and looking at the trend reveals whether differential
signal is concentrated at low / mid / high abundance.

In [ ]:
# [B-8b · MA plots — Blina vs Mock LFC vs mean abundance, per system, B cells, 6h]
ma_results_b = plot_ma_blina_vs_mock(
    adata,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='B',
)

## ΔLFC vs mean abundance — HT/NALM vs HT/HB, B cells, 6h

*LFC of LFC.* For each feature we compute the Blina − Mock LFC in each system
separately and then plot:

* **x**: mean signal across the two systems combined (Blina ∪ Mock pooled across
  HT/NALM and HT/HB), as the abundance baseline to standardise against.
* **y**: ΔLFC = LFC(HT/NALM) − LFC(HT/HB) — how much *more* the response in
  the NALM-6 system exceeds the response in the healthy primary B system.

Significant ΔLFC values at very low mean signal are flagged as low-confidence;
those at moderate / high mean signal are the system-specific responses worth
interpreting. Because the same healthy T donor is used in both systems, ΔLFC
isolates the contribution of the B target (NALM-6 line vs primary healthy B).

In [ ]:
# [B-8c · ΔLFC (LFC of LFC) vs mean signal — B cells, 6h, NALM vs HB]
# Color scheme mirrors [B-7 · LFC scatter]:
#   purple (#9467bd) = higher LFC in HT/NALM   (ΔLFC > 0)
#   green  (#2ca02c) = higher LFC in HT/HB     (ΔLFC < 0)
delta_ab_b, delta_sp_b = plot_delta_lfc_ma(
    ma_results_b,
    label_a='HT/NALM',
    label_b='HT/HB',
    cell_type='B',
    time_val='6h',
)

## Spatial subsets — B cells, 6h Blina

In [ ]:
# [B-9 · Spatial subsets — B cells, 6h Blina HT/NALM & HT/HB]
def _sp_subset_b(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'B') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building B-cell spatial subsets:')
sp_b_6h_blina_nm = _sp_subset_b(adata, '6h', 'Blinatumomab', SYS_HT_NALM)
sp_b_6h_blina_hb = _sp_subset_b(adata, '6h', 'Blinatumomab', SYS_HT_HB)

all_sp_cols_b = sp_b_6h_blina_nm.columns

## B cell APC synapse — HT/NALM vs HT/HB at 6h Blinatumomab

Antigen-presentation complex on the B-target side: MHC-II + costimulation +
inhibitory ligands + co-receptors + adhesion. Asks how the NALM-6 cell line
organises its synapse-facing surface relative to primary healthy B cells under
the same healthy T donor.

In [ ]:
# [B-12 · B cell APC synapse: heatmaps + networks at 6h Blina]
APC_SYNAPSE_CATEGORIES = {
    'MHC Class II (Ag presentation)':  (['HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC'], '#e41a1c'),
    'Costimulation':                    (['CD80', 'CD86', 'CD40'],                        '#ff7f00'),
    'Inhibitory / Checkpoint ligands':  (['CD274', 'CD273', 'CD32', 'CD72', 'CD305'],     '#377eb8'),
    'B cell co-receptors':              (['CD19', 'CD20', 'CD22', 'CD79a'],               '#4daf4a'),
    'Adhesion (pSMAC)':                 (['CD54', 'CD58', 'CD50', 'CD102'],               '#984ea3'),
}

mat_apc_nm, mat_apc_hb, mat_apc_diff = plot_synapse_suite(
    sp_a=sp_b_6h_blina_nm, sp_b=sp_b_6h_blina_hb,
    categories=APC_SYNAPSE_CATEGORIES,
    label_a='HT/NALM 6h Blina', label_b='HT/HB 6h Blina',
    diff_label='Diff (HT/NALM − HT/HB)',
    suite_name='APC synapse',
    highlight_node='HLA-DR-DP-DQ',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)

# Inhibitory signals — HT/NALM vs HT/HB

Focused look at inhibitory / checkpoint biology on both sides of the synapse:

* **CD8 T side:** TIM-3 (CD366), TIGIT, VISTA, PSGL-1 (CD162), PD-1 (CD279).
* **B target side:** VISTA, CEACAM8 (CD66b), PD-L1 (CD274), PD-L2 (CD273).

Sections below mirror the rest of the notebook but restrict the marker universe
to **inhibitory ∪ relevant-synapse**:

1. Per-panel violins across time × condition (`[Inh-2T]`, `[Inh-2B]`).
2. Per-system Blina-vs-Mock abundance + spatial top-N (`[Inh-8T]`, `[Inh-8B]`).
3. MA plots (LFC vs mean) per system (`[Inh-8bT]`, `[Inh-8bB]`).
4. ΔLFC (HT/NALM − HT/HB) vs mean signal (`[Inh-8cT]`, `[Inh-8cB]`).

In [ ]:
# [Inh-0 · Marker scope — abundance = inhibitory only; spatial pairs require ≥1 inhibitory endpoint]
INHIB_T = ['CD366', 'TIGIT', 'VISTA', 'CD162', 'CD279']           # TIM-3, TIGIT, VISTA, PSGL-1, PD-1
INHIB_B = ['VISTA', 'CD66b', 'CD274', 'CD273']                    # VISTA, CEACAM8, PD-L1, PD-L2

# Pair partners are constrained to the inhibitory ∪ relevant-synapse universe
# (synapse lists reused from [12 · CD8 immune synapse] and [B-12 · B APC synapse]).
T_SYNAPSE = ['CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137', 'CD226',
             'TIGIT', 'CD279', 'VISTA',
             'CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48', 'CD352', 'CD53',
             'CD45', 'CD43', 'CD44']
B_SYNAPSE = ['HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC',
             'CD80', 'CD86', 'CD40',
             'CD274', 'CD273', 'CD32', 'CD72', 'CD305',
             'CD19', 'CD20', 'CD22', 'CD79a',
             'CD54', 'CD58', 'CD50', 'CD102']

T_PAIR_SCOPE = sorted(set(INHIB_T) | set(T_SYNAPSE))
B_PAIR_SCOPE = sorted(set(INHIB_B) | set(B_SYNAPSE))

def _scope_adata(adata_in, abundance_markers, pair_scope, inhib_markers,
                 obsm_key='spatial_asinh5'):
    """Subset adata var_names to *abundance_markers* (inhibitory only); keep obsm
    pairs whose endpoints are in *pair_scope* AND where ≥1 endpoint is inhibitory."""
    keep = [m for m in abundance_markers if m in adata_in.var_names]
    sub = adata_in[:, keep].copy()
    sp = sub.obsm[obsm_key]
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    pair_set = set(pair_scope)
    inhib_set = set(inhib_markers)
    cols = []
    for c in sp.columns:
        endpoints = c.split('/')
        if all(m in pair_set for m in endpoints) and any(m in inhib_set for m in endpoints):
            cols.append(c)
    sub.obsm[obsm_key] = sp[cols].copy()
    return sub

adata_cd8_scope = _scope_adata(adata, INHIB_T, T_PAIR_SCOPE, INHIB_T)
adata_b_scope   = _scope_adata(adata, INHIB_B, B_PAIR_SCOPE, INHIB_B)

print(f'CD8 scope: abundance {adata_cd8_scope.n_vars} markers '
      f'({list(adata_cd8_scope.var_names)}), '
      f'{adata_cd8_scope.obsm["spatial_asinh5"].shape[1]} spatial pairs (≥1 inhibitory endpoint)')
print(f'B   scope: abundance {adata_b_scope.n_vars} markers '
      f'({list(adata_b_scope.var_names)}), '
      f'{adata_b_scope.obsm["spatial_asinh5"].shape[1]} spatial pairs (≥1 inhibitory endpoint)')

## Cross-condition comparison — inhibitory markers across time × condition

In [ ]:
# [Inh-2T · 4-way DA panel: CD8 inhibitory markers — HT/NALM vs HT/HB across time × condition]
mask_nm_cd8 = (
    (adata.obs['cell_system'] == SYS_HT_NALM) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_nm_cd8_inh = adata[mask_nm_cd8].copy()
adata_nm_cd8_inh.obs['time_cond'] = (
    adata_nm_cd8_inh.obs['time'].astype(str) + ' ' + adata_nm_cd8_inh.obs['condition'].astype(str)
)

mask_hb_cd8 = (
    (adata.obs['cell_system'] == SYS_HT_HB) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_hb_cd8_inh = adata[mask_hb_cd8].copy()
adata_hb_cd8_inh.obs['time_cond'] = (
    adata_hb_cd8_inh.obs['time'].astype(str) + ' ' + adata_hb_cd8_inh.obs['condition'].astype(str)
)

print(f'CD8 HT/NALM: {adata_nm_cd8_inh.n_obs}')
print(adata_nm_cd8_inh.obs['time_cond'].value_counts().to_string())
print(f'\nCD8 HT/HB: {adata_hb_cd8_inh.n_obs}')
print(adata_hb_cd8_inh.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_nm_cd8_inh, 't_inhibitory_markers',
    group_key='time_cond',
    adata_compare=adata_hb_cd8_inh,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

In [ ]:
# [Inh-2B · 4-way DA panel: B cell inhibitory markers — HT/NALM vs HT/HB across time × condition]
mask_nm_b = (
    (adata.obs['cell_system'] == SYS_HT_NALM) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_nm_b_inh = adata[mask_nm_b].copy()
adata_nm_b_inh.obs['time_cond'] = (
    adata_nm_b_inh.obs['time'].astype(str) + ' ' + adata_nm_b_inh.obs['condition'].astype(str)
)

mask_hb_b = (
    (adata.obs['cell_system'] == SYS_HT_HB) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_hb_b_inh = adata[mask_hb_b].copy()
adata_hb_b_inh.obs['time_cond'] = (
    adata_hb_b_inh.obs['time'].astype(str) + ' ' + adata_hb_b_inh.obs['condition'].astype(str)
)

print(f'B HT/NALM: {adata_nm_b_inh.n_obs}')
print(adata_nm_b_inh.obs['time_cond'].value_counts().to_string())
print(f'\nB HT/HB: {adata_hb_b_inh.n_obs}')
print(adata_hb_b_inh.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_nm_b_inh, 'b_inhibitory_markers',
    group_key='time_cond',
    adata_compare=adata_hb_b_inh,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

## Mock vs Blinatumomab — abundance + spatial, inhibitory ∪ synapse, per system

Same 2×2 layout as [8] / [B-8] but with the marker universe restricted to
inhibitory checkpoints plus the immune-synapse / APC-synapse panels used in
[12] / [B-12]. Spatial pairs are kept only when **both** endpoints lie inside
the scope, so colocalization bars reflect inhibitory–inhibitory and
inhibitory–synapse interactions.

In [ ]:
# [Inh-8T · CD8 Blina vs Mock — abundance + spatial, inhibitory ∪ synapse, per system, 6h]
for sys_label, sys_val in [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)]:
    plot_condition_comparison(
        adata_cd8_scope,
        time_val='6h',
        cell_system=sys_val,
        cell_type='CD8',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

In [ ]:
# [Inh-8B · B cell Blina vs Mock — abundance + spatial, inhibitory ∪ synapse, per system, 6h]
for sys_label, sys_val in [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)]:
    plot_condition_comparison(
        adata_b_scope,
        time_val='6h',
        cell_system=sys_val,
        cell_type='B',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## MA plots — inhibitory ∪ synapse, 6h

MA-style LFC vs mean signal, restricted to the inhibitory ∪ synapse marker
universe (spatial pairs kept only if both endpoints are inside the scope).

In [ ]:
# [Inh-8bT · MA plots — Blina vs Mock, CD8, inhibitory ∪ synapse, 6h]
ma_results_cd8_inh = plot_ma_blina_vs_mock(
    adata_cd8_scope,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='CD8',
)

In [ ]:
# [Inh-8bB · MA plots — Blina vs Mock, B cells, inhibitory ∪ synapse, 6h]
ma_results_b_inh = plot_ma_blina_vs_mock(
    adata_b_scope,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='B',
)

## LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), inhibitory markers, 6h

Per-marker Blina/Mock LFC for the two systems, plotted system-against-system
(same view as [7] / [B-7]). Restricted to the inhibitory markers via
`adata_cd8_scope` / `adata_b_scope` — so only TIM-3/TIGIT/VISTA/CD162/PD-1 (CD8)
and VISTA/CD66b/CD274/CD273 (B) are shown. Points above y=x have a stronger
response in HT/NALM (purple); points below have a stronger response in HT/HB
(green).

In [ ]:
# [Inh-8cT · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), CD8 inhibitory, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata_cd8_scope, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD8',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',  # higher LFC in HT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in HT/HB   (below y=x)
    top_k=5,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# [Inh-8cB · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), B inhibitory, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata_b_scope, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='B',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',  # higher LFC in HT/NALM
    color_below='#2ca02c',  # higher LFC in HT/HB
    top_k=4,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), inhibitory pairs, 6h

Same system-vs-system scatter as the abundance LFC plot above, but applied to
the **spatial colocalization** pairs scoped to ≥1 inhibitory endpoint. Mean
difference (Blina − Mock) is used rather than log2 fold change (coloc values
can be negative for repulsion). Points above y=x had a stronger Blina-induced
colocalization shift in HT/NALM; below y=x, stronger in HT/HB.

In [ ]:
# [Inh-8eT · Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), CD8 inhibitory pairs, 6h]
def plot_spatial_diff_scatter(
    adata_scope, time_val, cond_num, cond_den, sys_a, sys_b, cell_type,
    label_a, label_b, color_above, color_below,
    top_k=10, obsm_key='spatial_asinh5', ax=None,
):
    """Per-pair (cond_num − cond_den) spatial mean-diff scatter, sys_a (x) vs sys_b (y).

    Mirrors plot_lfc_scatter but on obsm spatial pairs using a mean-difference
    (not log2 ratio, since coloc values can be negative for repulsion).
    """
    def _diff(system_val):
        mask = (
            (adata_scope.obs['time'] == time_val) &
            (adata_scope.obs['cell_type_annot'] == cell_type) &
            (adata_scope.obs['cell_system'] == system_val)
        )
        sub = adata_scope[mask]
        sp = sub.obsm[obsm_key]
        if not isinstance(sp, pd.DataFrame):
            sp = pd.DataFrame(sp, index=sub.obs_names)
        cond = sub.obs['condition'].values
        num = sp.values[cond == cond_num].mean(axis=0)
        den = sp.values[cond == cond_den].mean(axis=0)
        return pd.Series(num - den, index=sp.columns)

    a = _diff(sys_a)
    b = _diff(sys_b)
    df = pd.DataFrame({'a': a, 'b': b}).replace([np.inf, -np.inf], np.nan).dropna()
    df['off_diag'] = (df['b'] - df['a']) / np.sqrt(2)

    top_idx = df['off_diag'].abs().nlargest(top_k).index
    colors = np.where(df.loc[top_idx, 'off_diag'] > 0, color_above, color_below)
    color_map = dict(zip(top_idx, colors))

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 8))

    lim = max(df[['a', 'b']].abs().max().max() * 1.15, 0.1)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='grey', lw=0.7, ls=':')

    other_idx = df.index.difference(top_idx)
    ax.scatter(df.loc[other_idx, 'a'], df.loc[other_idx, 'b'],
               s=30, alpha=0.55, color='#bbbbbb', edgecolor='white', linewidth=0.5)
    ax.scatter(df.loc[top_idx, 'a'], df.loc[top_idx, 'b'],
               s=70, alpha=0.9, c=[color_map[m] for m in top_idx],
               edgecolor='black', linewidth=0.6, zorder=3)

    def _pair_label(p):
        x, y = split_pair(p)
        return f'{display_name(x)} / {display_name(y)}'

    for m in top_idx:
        ax.annotate(_pair_label(m), (df.loc[m, 'a'], df.loc[m, 'b']),
                    fontsize=9, fontweight='bold', color=color_map[m],
                    xytext=(4, 4), textcoords='offset points')

    r = df['a'].corr(df['b'])
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(f'Mean diff {cond_num} − {cond_den}\n{label_a}')
    ax.set_ylabel(f'Mean diff {cond_num} − {cond_den}\n{label_b}')
    ax.set_title(f'{cell_type} spatial pairs — {time_val}   (n={len(df)}, Pearson r={r:.2f})')

    print(f'\n{"=" * 65}')
    print(f'  {cell_type} {time_val}  —  top {top_k} pairs farthest from y=x')
    print(f'  {color_above} = higher diff in {label_b} | {color_below} = higher diff in {label_a}')
    print(f'{"=" * 65}')
    top_tbl = df.loc[top_idx, ['a', 'b', 'off_diag']].copy()
    top_tbl.index = [_pair_label(m) for m in top_tbl.index]
    top_tbl.columns = [f'diff_{label_a}', f'diff_{label_b}', 'off_diag']
    top_tbl = top_tbl.reindex(top_tbl['off_diag'].abs().sort_values(ascending=False).index)
    print(top_tbl.round(3).to_string())
    return df


fig, ax = plt.subplots(figsize=(8, 8))
plot_spatial_diff_scatter(
    adata_cd8_scope, time_val='6h',
    cond_num='Blinatumomab', cond_den='Mock',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD8',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd', color_below='#2ca02c',
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# [Inh-8eB · Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), B inhibitory pairs, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_spatial_diff_scatter(
    adata_b_scope, time_val='6h',
    cond_num='Blinatumomab', cond_den='Mock',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='B',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd', color_below='#2ca02c',
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Inhibitory coloc heatmaps + networks — HT/NALM vs HT/HB at 6h Blinatumomab

Same `plot_synapse_suite` view as [12] / [B-12] but restricted to the inhibitory
panels: CD8 side (TIM-3, TIGIT, VISTA, PSGL-1, PD-1) and B side (PSGL-1,
CEACAM8, PD-L1, PD-L2). Three heatmaps + three force-directed networks per
suite — HT/NALM, HT/HB, and the difference. Uses the existing
`sp_6h_blina_nm` / `sp_6h_blina_hb` (CD8) and `sp_b_6h_blina_nm` /
`sp_b_6h_blina_hb` (B) spatial subsets built earlier in the notebook.

In [ ]:
# [Inh-12T · CD8 inhibitory coloc heatmap + network — HT/NALM vs HT/HB, 6h Blina]
T_INHIBITORY_CATEGORIES = {
    'Co-inhibitory checkpoints': (['CD366', 'TIGIT', 'CD279'], '#e41a1c'),  # TIM-3, TIGIT, PD-1
    'B7 family (VISTA)':         (['VISTA'],                   '#377eb8'),
    'Selectin ligand':           (['CD162'],                   '#4daf4a'),  # PSGL-1
}

mat_inhT_nm, mat_inhT_hb, mat_inhT_diff = plot_synapse_suite(
    sp_a=sp_6h_blina_nm, sp_b=sp_6h_blina_hb,
    categories=T_INHIBITORY_CATEGORIES,
    label_a='HT/NALM 6h Blina', label_b='HT/HB 6h Blina',
    diff_label='Diff (HT/NALM − HT/HB)',
    suite_name='CD8 inhibitory',
    highlight_node='CD279',
    var_filter=adata.var_names,
    cluster_k=[2, 3],
)

In [ ]:
# [Inh-12B · B cell inhibitory coloc heatmap + network — HT/NALM vs HT/HB, 6h Blina]
B_INHIBITORY_CATEGORIES = {
    'PD-L axis':            (['CD274', 'CD273'], '#e41a1c'),    # PD-L1, PD-L2
    'B7 family (VISTA)':    (['VISTA'],          '#377eb8'),
    'Granulocyte / CEACAM': (['CD66b'],          '#ff7f00'),    # CEACAM8
}

mat_inhB_nm, mat_inhB_hb, mat_inhB_diff = plot_synapse_suite(
    sp_a=sp_b_6h_blina_nm, sp_b=sp_b_6h_blina_hb,
    categories=B_INHIBITORY_CATEGORIES,
    label_a='HT/NALM 6h Blina', label_b='HT/HB 6h Blina',
    diff_label='Diff (HT/NALM − HT/HB)',
    suite_name='B inhibitory',
    highlight_node='CD274',
    var_filter=adata.var_names,
    cluster_k=[2, 3],
)

# CD39 / CD73 — Breg ectoenzymes on B cells

Short focused section: adenosine-generating ectoenzymes CD39 (ENTPD1) and CD73
(NT5E) on the B target compartment. Violins across time × condition, abundance
LFC scatter (HT/HB vs HT/NALM), spatial coloc diff scatter for pairs involving
CD39/CD73, and per-system MA plots. Depends on `adata_nm_b_inh` / `adata_hb_b_inh`
(from `[Inh-2B]`) and the helpers `_scope_adata`, `plot_spatial_diff_scatter`
defined earlier in the inhibitory section.

In [ ]:
# [Breg-0 · Marker scope — B cells, CD39 / CD73; spatial pairs restricted to CD39/CD73 only]
BREG_B = ['CD39', 'CD73']
adata_b_breg = _scope_adata(
    adata,
    abundance_markers=BREG_B,
    pair_scope=BREG_B,        # both endpoints must be in {CD39, CD73}
    inhib_markers=BREG_B,
)
print(f'B Breg scope: abundance {adata_b_breg.n_vars} markers '
      f'({list(adata_b_breg.var_names)}), '
      f'{adata_b_breg.obsm["spatial_asinh5"].shape[1]} spatial pairs '
      f'({list(adata_b_breg.obsm["spatial_asinh5"].columns)})')

In [ ]:
# [Breg-2 · Violins: CD39 / CD73 on B cells — HT/NALM vs HT/HB across time × condition]
plot_marker_panel_violins(
    adata_nm_b_inh, 'b_breg_markers',
    group_key='time_cond',
    adata_compare=adata_hb_b_inh,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

In [ ]:
# [Breg-8c · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), B cells CD39/CD73, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata_b_breg, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='B',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd', color_below='#2ca02c',
    top_k=2,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# [Breg-8e · Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), CD39/CD73 pairs, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_spatial_diff_scatter(
    adata_b_breg, time_val='6h',
    cond_num='Blinatumomab', cond_den='Mock',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='B',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd', color_below='#2ca02c',
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# [Breg-8b · MA plots — Blina vs Mock, B cells, CD39/CD73 + partners, 6h]
ma_results_breg = plot_ma_blina_vs_mock(
    adata_b_breg,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='B',
)